# 1. Carregamento dos dados

In [135]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("janiobachmann/bank-marketing-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'bank-marketing-dataset' dataset.
Path to dataset files: /kaggle/input/bank-marketing-dataset


In [136]:
import pandas as pd

df = pd.read_csv('/kaggle/input/bank-marketing-dataset/bank.csv')


# 2. Exploração inicial dos dados
## Aqui verificamos as primeiras linhas e os tipos de dados para entender a estrutura da base.

In [137]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes
5,42,management,single,tertiary,no,0,yes,yes,unknown,5,may,562,2,-1,0,unknown,yes
6,56,management,married,tertiary,no,830,yes,yes,unknown,6,may,1201,1,-1,0,unknown,yes
7,60,retired,divorced,secondary,no,545,yes,no,unknown,6,may,1030,1,-1,0,unknown,yes
8,37,technician,married,secondary,no,1,yes,no,unknown,6,may,608,1,-1,0,unknown,yes
9,28,services,single,secondary,no,5090,yes,no,unknown,6,may,1297,3,-1,0,unknown,yes


In [138]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11162 non-null  int64 
 1   job        11162 non-null  object
 2   marital    11162 non-null  object
 3   education  11162 non-null  object
 4   default    11162 non-null  object
 5   balance    11162 non-null  int64 
 6   housing    11162 non-null  object
 7   loan       11162 non-null  object
 8   contact    11162 non-null  object
 9   day        11162 non-null  int64 
 10  month      11162 non-null  object
 11  duration   11162 non-null  int64 
 12  campaign   11162 non-null  int64 
 13  pdays      11162 non-null  int64 
 14  previous   11162 non-null  int64 
 15  poutcome   11162 non-null  object
 16  deposit    11162 non-null  object
dtypes: int64(7), object(10)
memory usage: 1.4+ MB


# 2. Limpeza e preparação dos dados
## Após verificar se Dtype mostrado na info está conforme esperado, verificamos nulos e mapeamos target

In [139]:
df.isnull().sum()

,0
age,0
job,0
marital,0
education,0
default,0
balance,0
housing,0
loan,0
contact,0
day,0


In [140]:
df['deposit'] = df['deposit'].map({'yes':1, 'no':0})

# 3. Criação de X e y
## Aqui criamos nossas variáveis, transformando categóricas, removendo colunas como duration(leakeage)

In [141]:
X = df.drop(['deposit', 'duration'], axis=1)

In [142]:
X = pd.get_dummies(X)

In [143]:
y = df['deposit']

In [144]:
X.shape

(11162, 50)

# 4. Treinamento do modelo

##Modelo Base
### Iniciamos criando modelo base para posterior avaliação e comparação

In [145]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [146]:
from sklearn.tree import DecisionTreeClassifier

modelo_base = DecisionTreeClassifier(max_depth=5, random_state=42)

In [147]:
modelo_base.fit(X_treino, y_treino)

DecisionTreeClassifier(max_depth=5, random_state=42)

In [148]:
pred_base = modelo_base.predict(X_teste)

In [149]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_teste, pred_base))
print(classification_report(y_teste, pred_base))

[[1474  268]
 [ 775  832]]
              precision    recall  f1-score   support

           0       0.66      0.85      0.74      1742
           1       0.76      0.52      0.61      1607

    accuracy                           0.69      3349
   macro avg       0.71      0.68      0.68      3349
weighted avg       0.70      0.69      0.68      3349



## Modelo Balanceado

In [150]:
modelo_balanced = DecisionTreeClassifier(max_depth=5, class_weight='balanced')

In [151]:
modelo_balanced.fit(X_treino, y_treino)

DecisionTreeClassifier(class_weight='balanced', max_depth=5)

In [152]:
pred_balanced = modelo_balanced.predict(X_teste)

In [153]:
print(confusion_matrix(y_teste, pred_balanced))
print(classification_report(y_teste, pred_balanced))

[[1373  369]
 [ 678  929]]
              precision    recall  f1-score   support

           0       0.67      0.79      0.72      1742
           1       0.72      0.58      0.64      1607

    accuracy                           0.69      3349
   macro avg       0.69      0.68      0.68      3349
weighted avg       0.69      0.69      0.68      3349



# 5. Otimização com threshold e lucro

In [154]:
probs = modelo_base.predict_proba(X_teste)[:,1]

In [155]:
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.1)

for t in thresholds:
    pred = (probs >= t).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_teste, pred).ravel()

    lucro = (tp * 1000) - ((tp + fp) * 50)

    print(f"Threshold {t:.1f} → Lucro: {lucro}")

Threshold 0.1 → Lucro: 1437750
Threshold 0.2 → Lucro: 1378100
Threshold 0.3 → Lucro: 1315950
Threshold 0.4 → Lucro: 864100
Threshold 0.5 → Lucro: 777000
Threshold 0.6 → Lucro: 776450
Threshold 0.7 → Lucro: 347000
Threshold 0.8 → Lucro: 347000


# 📊 Resultado Final
O threshold 0.1 gerou maior lucro,
indicando que uma estratégia agressiva é mais vantajosa,
pois o valor da venda supera o custo da campanha.

# 📌 Conclusão
O modelo inicial apresentou desempenho equilibrado,
mas com perda significativa de clientes potenciais.

Ao ajustar o threshold e priorizar recall,
foi possível aumentar o lucro da campanha,
mesmo com maior custo operacional.

Isso demonstra que decisões baseadas em métricas de negócio
são mais relevantes do que métricas tradicionais como acurácia.